# PaceAlgo ML — Notebook 12: Model Battery

Vergleich von 4 Modellarchitekturen auf der **besten Feature-Konfiguration aus NB 11** (`phase1_best_config.json`):

1. **LightGBM** (Pine-budget: 30 trees, depth 3) — bisheriges baseline
2. **XGBoost** (gleiche depth + trees) — kann marginal andere Splits finden
3. **CatBoost** (gleiche) — bessere categorical handling, oft robusterer Default
4. **Voting Ensemble** — Mittelwert der 3 calibrated probabilities

**Strikte Methodik:**
- Identische Train/Val/Test-Splits (Walk-Forward)
- Identische Tier-Cutoffs aus VAL-Set abgeleitet
- Pro Modell × Tier: PF / WR / Expectancy / Stability-CV
- Pine-Export-Eignung berücksichtigt (LGBM/XGB exportierbar; CatBoost komplex)

**Entscheidungslogik:**
- Bestes Modell muss `Premium-Tier-PF > NB-11-Bestwert + 0.05` haben um den Wechsel zu rechtfertigen
- Sonst bleibt LightGBM (einfachster Pine-Export)

## 1. Setup

In [ ]:
import sys, os
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    DATA_PROCESSED = Path('/content/processed')
    DATA_EXT = Path('/content/processed_v2')
    ARTIFACTS = Path(PROJECT_ROOT) / 'artifacts'
    REPORTS_DIR = ARTIFACTS / 'reports'
    MODELS_DIR = ARTIFACTS / 'models'
    DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    os.chdir(PROJECT_ROOT)
    !rm -rf /tmp/pace-algo
    !git clone -q https://github.com/ecoNC/pace-algo.git /tmp/pace-algo
    !cp -rf /tmp/pace-algo/core/* {PROJECT_ROOT}/core/
    print('Core code updated from GitHub')
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

    # CRITICAL: restore labels + base features from Drive if local is empty
    # (these are needed when NB 12 rebuilds extended features in Section 2.5)
    n_local_labels = len(list(DATA_PROCESSED.glob('labels_*.parquet')))
    n_drive_labels = len(list(DRIVE_BACKUP.glob('labels_*.parquet'))) if DRIVE_BACKUP.exists() else 0
    print(f'Labels in local: {n_local_labels}, Drive backup: {n_drive_labels}')
    if n_local_labels < n_drive_labels:
        print('Restoring labels + base features from Drive...')
        !rsync -ah {DRIVE_BACKUP}/ {DATA_PROCESSED}/
        n_local_labels = len(list(DATA_PROCESSED.glob('labels_*.parquet')))
        print(f'After restore: {n_local_labels} label files in /content/processed/')
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)
    from core.config import DATA_PROCESSED, ARTIFACTS_REPORTS, ARTIFACTS_MODELS
    REPORTS_DIR = ARTIFACTS_REPORTS
    MODELS_DIR = ARTIFACTS_MODELS
    DATA_EXT = DATA_PROCESSED.parent / 'processed_v2'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]


In [ ]:
!pip install -q lightgbm xgboost catboost 2>&1 | tail -1

import json
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from tqdm.notebook import tqdm
from datetime import datetime, timezone

from core.config import (
    FX_SYMBOLS, METAL_SYMBOLS, PRIMARY_TIMEFRAMES, DEV_HOLDOUT_SYMBOLS,
    TRAIN_END, VAL_END, HTF_CONTEXT_TIMEFRAMES,
)
from core.features import (
    compute_features, attach_macro, attach_htf_context,
    compute_smc_features, compute_session_features, compute_htf_interactions,
)
from core.features.engineer import atr as atr_fn
from core.train import walk_forward_split, binary_label_for_long

# Reproducibility — single source of truth for all 3 models + ensemble
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Results export directory — written at end of notebook (Section 10)
RESULTS_DIR = Path(PROJECT_ROOT) / 'results'
for sub in ('json_exports', 'benchmark_tables', 'walk_forward_summaries',
            'per_symbol_metrics', 'yearly_stability_tables'):
    (RESULTS_DIR / sub).mkdir(parents=True, exist_ok=True)
RUN_DATE = datetime.now(timezone.utc).strftime('%Y-%m-%d')

# Path setup — match NB 11
if IS_COLAB:
    DATA_RAW = Path(PROJECT_ROOT) / 'data_cache' / 'raw'
else:
    from core.config import DATA_RAW

R_VALUE = 1.5
WIN_R = 1.5
LOSS_R = 1.0
print('Libraries loaded:')
print(f'  LightGBM: {lgb.__version__}')
print(f'  XGBoost:  {xgb.__version__}')
import catboost
print(f'  CatBoost: {catboost.__version__}')
print(f'  Random seed (all models): {RANDOM_SEED}')
print(f'  Results dir: {RESULTS_DIR}')
print(f'  Run date (UTC): {RUN_DATE}')

# Make sure extended directory exists
DATA_EXT.mkdir(parents=True, exist_ok=True)
print(f'\nExtended features dir: {DATA_EXT}')

# Check if extended files exist (may have been wiped between NB 11 and NB 12)
n_existing = len(list(DATA_EXT.glob('*_extended.parquet')))
print(f'Extended feature files present: {n_existing}')

# Drive backup location (in case we sync later)
if IS_COLAB:
    EXT_DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed_v2'
    EXT_DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)
    n_backup = len(list(EXT_DRIVE_BACKUP.glob('*_extended.parquet')))
    print(f'Drive backup files: {n_backup}')
    if n_existing < n_backup:
        print('Restoring extended features from Drive backup...')
        !rsync -ah {EXT_DRIVE_BACKUP}/ {DATA_EXT}/
        n_existing = len(list(DATA_EXT.glob('*_extended.parquet')))
        print(f'After restore: {n_existing} files')


## 2. Load Best Config from NB 11

In [ ]:
# NB 11 best config — saved as artifacts/reports/phase1_best_config.json
# Fallback: hardcoded from NB 11 winning configuration (FX-only, 27 features)
HARDCODED_FALLBACK = {
    'experiment_name': 'asset_FX_only',
    'asset_scope':     'FX_only',
    'feature_cols': [
        'hour_sin', 'dist_to_swing_low_atr', 'htf_1h_rsi_14', 'realized_vol_20',
        'atr_percentile_100', 'atr_pct', 'dist_to_swing_high_atr', 'volume_z_score',
        'ema_20_slope_atr', 'hour_cos', 'momentum_composite', 'rvol_20',
        'adx_14', 'ema_20_dist_atr', 'htf_1h_atr_percentile_100',
        'htf_ltf_agree_bull', 'htf_ltf_agree_bear', 'htf_ltf_counter_trend',
        'htf_ltf_alignment_score', 'ltf_rsi_minus_htf_rsi',
        'both_rsi_oversold', 'both_rsi_overbought', 'vol_pct_diff_htf',
        'both_high_vol', 'both_low_vol', 'pullback_in_bull', 'pullback_in_bear',
    ],
    'n_features': 27,
    'premium_pf_oos': 2.015,
    'premium_wr_oos': 0.573,
    'premium_expR_oos': 0.4264,
}

config_path = REPORTS_DIR / 'phase1_best_config.json'
if config_path.exists():
    with open(config_path) as f:
        best_config = json.load(f)
    print(f'Loaded best config from {config_path}')
else:
    best_config = HARDCODED_FALLBACK
    print(f'phase1_best_config.json not found — using HARDCODED fallback (NB 11 winning config)')

print(f'\n  experiment_name: {best_config["experiment_name"]}')
print(f'  asset_scope:     {best_config["asset_scope"]}')
print(f'  n_features:      {best_config["n_features"]}')
print(f'  PF (NB11):       {best_config["premium_pf_oos"]:.4f}')
print(f'  WR (NB11):       {best_config["premium_wr_oos"]*100:.1f}%')
print(f'  ExpR (NB11):     {best_config["premium_expR_oos"]:+.4f}')
print(f'  features: {best_config["feature_cols"][:5]} ... (+{len(best_config["feature_cols"])-5} more)')


## 2.5 Rebuild Extended Features if Missing

NB 11 baut diese in `/content/processed_v2/` — Colab löscht /content/ bei Runtime-Wechsel.
Diese Cell baut sie aus `data_cache/raw/` neu auf, falls sie fehlen.


In [ ]:
def load_ohlcv_raw(symbol, tf):
    p = DATA_RAW / f'{symbol}_{tf}.parquet'
    return pd.read_parquet(p) if p.exists() else None

# Validate: do existing extended files have the 'label' column?
# If not, they were built without labels (NB 11 didn't have labels available) — delete and rebuild
def has_label_column(path):
    try:
        # Read only first few rows to check schema
        df_check = pd.read_parquet(path, columns=None).head(1)
        return 'label' in df_check.columns
    except Exception:
        return False

existing_files = list(DATA_EXT.glob('*_extended.parquet'))
if existing_files:
    sample_path = existing_files[0]
    if not has_label_column(sample_path):
        print(f'WARNING: existing extended files missing \"label\" column.')
        print(f'  Deleting {len(existing_files)} stale files to force rebuild with labels...')
        for f in existing_files:
            f.unlink()
        # Also clear Drive backup so it doesn't restore broken files
        if IS_COLAB and EXT_DRIVE_BACKUP.exists():
            for f in EXT_DRIVE_BACKUP.glob('*_extended.parquet'):
                f.unlink()
        print('  Stale files cleared.')

# Check what's needed
ALL_TRAIN_SYMBOLS = FX_SYMBOLS + METAL_SYMBOLS  # incl. GBPUSD even though dev-holdout
expected_files = {f'{s}_{tf}_extended.parquet' for s in ALL_TRAIN_SYMBOLS for tf in PRIMARY_TIMEFRAMES}
present_files = {p.name for p in DATA_EXT.glob('*_extended.parquet')}
missing = expected_files - present_files
print(f'Expected: {len(expected_files)} files, present: {len(present_files)}, missing: {len(missing)}')

if missing:
    print(f'\nBuilding missing extended features (this takes ~10-15 min)...')
    macro_path = DATA_RAW / 'macro_daily.parquet'
    macro = pd.read_parquet(macro_path) if macro_path.exists() else pd.DataFrame()

    n_label_files = len(list(DATA_PROCESSED.glob('labels_*.parquet')))
    if n_label_files == 0:
        print('  WARNING: no label files in DATA_PROCESSED. Extended features will be built WITHOUT labels.')
        print('  This will cause downstream failures. Run NB 04 first to generate labels.')

    for symbol in tqdm(ALL_TRAIN_SYMBOLS, desc='Symbols'):
        htf_feats = {}
        for htf in HTF_CONTEXT_TIMEFRAMES:
            d = load_ohlcv_raw(symbol, htf)
            htf_feats[htf] = compute_features(d) if (d is not None and not d.empty) else pd.DataFrame()

        for tf in PRIMARY_TIMEFRAMES:
            out_path = DATA_EXT / f'{symbol}_{tf}_extended.parquet'
            if out_path.exists():
                continue
            ohlcv = load_ohlcv_raw(symbol, tf)
            if ohlcv is None or ohlcv.empty:
                continue
            base = compute_features(ohlcv)
            base = attach_htf_context(base, htf_feats.get('1h', pd.DataFrame()), htf_feats.get('4h', pd.DataFrame()))
            base = attach_macro(base, macro)
            atr14 = atr_fn(ohlcv['high'], ohlcv['low'], ohlcv['close'], 14).values
            ema_align = base['ema_alignment'].fillna(0).values
            smc = compute_smc_features(ohlcv, atr14, ema_align)
            sess = compute_session_features(ohlcv, atr14)
            inter = compute_htf_interactions(base)
            ext = pd.concat([base, smc, sess, inter], axis=1)
            # Attach labels (from R=1.5 labeling) — MUST be present
            label_path = DATA_PROCESSED / f'labels_{symbol}_{tf}_R{R_VALUE}.parquet'
            if not label_path.exists():
                print(f'    SKIP {symbol} {tf}: labels missing at {label_path}')
                continue
            labels = pd.read_parquet(label_path)
            ext = ext.join(labels[['label']], how='inner')
            ext['symbol'] = symbol
            ext['timeframe'] = tf
            ext.to_parquet(out_path, compression='zstd')
        htf_feats.clear()

    n_now = len(list(DATA_EXT.glob('*_extended.parquet')))
    print(f'\nExtended features now ready: {n_now} files')
else:
    print('All extended features already present — skipping rebuild.')

# Sync to Drive backup so we don't have to rebuild next session
if IS_COLAB:
    print('\nSyncing extended features to Drive backup...')
    !rsync -ah {DATA_EXT}/ {EXT_DRIVE_BACKUP}/
    print(f'Drive backup updated.')


## 3. Load Extended Dataset (same as NB 11)

In [ ]:
def load_ext(sym, tf):
    p = DATA_EXT / f'{sym}_{tf}_extended.parquet'
    return pd.read_parquet(p) if p.exists() else None

scope = best_config['asset_scope']
if scope == 'FX_only':
    symbols = FX_SYMBOLS
elif scope == 'Gold_only':
    symbols = METAL_SYMBOLS
else:
    symbols = FX_SYMBOLS + METAL_SYMBOLS

drop = set(DEV_HOLDOUT_SYMBOLS)
frames = []
for s in symbols:
    if s in drop: continue
    for tf in PRIMARY_TIMEFRAMES:
        d = load_ext(s, tf)
        if d is not None and not d.empty:
            frames.append(d)
df_full = pd.concat(frames, axis=0).sort_index() if frames else pd.DataFrame()
print(f'Stacked dataset for scope "{scope}": {df_full.shape}')

features = best_config['feature_cols']
available = [f for f in features if f in df_full.columns]
print(f'Available features: {len(available)} of {len(features)}')

df_c = df_full.dropna(subset=available + ['label'])
train_df, val_df, test_df = walk_forward_split(df_c, TRAIN_END, VAL_END)
print(f'train: {len(train_df):,}  val: {len(val_df):,}  test: {len(test_df):,}')

X_tr = train_df[available].values; y_tr = binary_label_for_long(train_df['label']).values
X_vl = val_df[available].values; y_vl = binary_label_for_long(val_df['label']).values
X_ts = test_df[available].values; y_ts = binary_label_for_long(test_df['label']).values

print(f'\nClass balance — train: {y_tr.mean():.3f}, val: {y_vl.mean():.3f}, test: {y_ts.mean():.3f}')

## 4. Train 3 Models with Equivalent Hyperparameters

In [ ]:
# Common hyperparams: 30 trees, depth 3, mild L2 reg
# All 3 models use RANDOM_SEED (set in Section 1) for reproducibility
models = {}

print('Training LightGBM...')
lgb_params = {
    'objective': 'binary', 'metric': 'binary_logloss',
    'num_leaves': 7, 'max_depth': 3, 'min_data_in_leaf': 200,
    'learning_rate': 0.05, 'num_iterations': 30, 'lambda_l2': 0.5,
    'feature_fraction': 0.85, 'bagging_fraction': 0.85, 'bagging_freq': 5,
    'is_unbalance': True, 'verbose': -1, 'n_jobs': -1,
    'seed': RANDOM_SEED, 'deterministic': True,
}
td = lgb.Dataset(X_tr, label=y_tr)
vd = lgb.Dataset(X_vl, label=y_vl, reference=td)
models['LightGBM'] = lgb.train(lgb_params, td, num_boost_round=30,
                                  valid_sets=[vd], valid_names=['val'],
                                  callbacks=[lgb.log_evaluation(period=0)])
print('  LightGBM done')

In [ ]:
print('Training XGBoost...')
pos_weight = (1 - y_tr.mean()) / y_tr.mean()  # equivalent to is_unbalance
xgb_clf = xgb.XGBClassifier(
    n_estimators=30,
    max_depth=3,
    learning_rate=0.05,
    reg_lambda=0.5,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=pos_weight,
    use_label_encoder=False,
    eval_metric='logloss',
    n_jobs=-1,
    verbosity=0,
    random_state=RANDOM_SEED,
)
xgb_clf.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)
models['XGBoost'] = xgb_clf
print('  XGBoost done')

In [ ]:
print('Training CatBoost...')
cat_clf = CatBoostClassifier(
    iterations=30,
    depth=3,
    learning_rate=0.05,
    l2_leaf_reg=3.0,
    rsm=0.85,
    auto_class_weights='Balanced',
    verbose=False,
    thread_count=-1,
    random_seed=RANDOM_SEED,
)
cat_clf.fit(X_tr, y_tr, eval_set=(X_vl, y_vl), use_best_model=False)
models['CatBoost'] = cat_clf
print('  CatBoost done')

print('\nAll 3 base models trained.')

## 5. Predict + Voting Ensemble

In [ ]:
val_probas = {}
test_probas = {}

val_probas['LightGBM'] = models['LightGBM'].predict(X_vl)
test_probas['LightGBM'] = models['LightGBM'].predict(X_ts)

val_probas['XGBoost'] = models['XGBoost'].predict_proba(X_vl)[:, 1]
test_probas['XGBoost'] = models['XGBoost'].predict_proba(X_ts)[:, 1]

val_probas['CatBoost'] = models['CatBoost'].predict_proba(X_vl)[:, 1]
test_probas['CatBoost'] = models['CatBoost'].predict_proba(X_ts)[:, 1]

# Voting ensemble = average
val_probas['Voting'] = (val_probas['LightGBM'] + val_probas['XGBoost'] + val_probas['CatBoost']) / 3
test_probas['Voting'] = (test_probas['LightGBM'] + test_probas['XGBoost'] + test_probas['CatBoost']) / 3

print('All probas computed.')
for name in val_probas:
    print(f'  {name}: VAL range [{val_probas[name].min():.3f}, {val_probas[name].max():.3f}]')

## 6. Per-Model × Per-Tier Evaluation

In [ ]:
def tier_metrics(labels):
    wins = int((labels == 1).sum())
    losses = int((labels == -1).sum())
    neutrals = int((labels == 0).sum())
    total = wins + losses + neutrals
    pf = (wins * WIN_R) / (losses * LOSS_R) if losses > 0 else (float('inf') if wins > 0 else 0.0)
    wr = wins / (wins + losses) if (wins + losses) > 0 else 0.0
    expR = (wins * WIN_R - losses * LOSS_R) / total if total > 0 else 0.0
    return {'n': total, 'wins': wins, 'losses': losses, 'wr': wr, 'pf': pf, 'expR': expR}

rows = []
for model_name in ['LightGBM', 'XGBoost', 'CatBoost', 'Voting']:
    vp = val_probas[model_name]
    tp = test_probas[model_name]
    vs = np.sort(vp)[::-1]
    cutoff_std = float(vs[max(1, int(len(vs) * 0.10) - 1)])
    cutoff_high = float(vs[max(1, int(len(vs) * 0.03) - 1)])
    cutoff_prem = float(vs[max(1, int(len(vs) * 0.01) - 1)])
    # AUC on TEST for reference
    try:
        test_auc = float(roc_auc_score(y_ts, tp))
    except Exception:
        test_auc = float('nan')
    for tier_name, cutoff in [('Standard', cutoff_std), ('High', cutoff_high), ('Premium', cutoff_prem)]:
        mask = tp >= cutoff
        sub_labels = test_df['label'].iloc[mask.nonzero()[0]]
        m = tier_metrics(sub_labels)
        m['model'] = model_name
        m['tier'] = tier_name
        m['cutoff'] = cutoff
        m['auc'] = test_auc
        rows.append(m)

battery_df = pd.DataFrame(rows)

print('=== Profit Factor by model × tier ===')
display(battery_df.pivot_table(index='model', columns='tier', values='pf').round(4)[['Standard', 'High', 'Premium']])

print('\n=== Win Rate ===')
display(battery_df.pivot_table(index='model', columns='tier', values='wr').round(4)[['Standard', 'High', 'Premium']])

print('\n=== Expectancy per Trade ===')
display(battery_df.pivot_table(index='model', columns='tier', values='expR').round(4)[['Standard', 'High', 'Premium']])

print('\n=== Trade Count ===')
display(battery_df.pivot_table(index='model', columns='tier', values='n').round(0)[['Standard', 'High', 'Premium']])

print('\n=== TEST AUC per model ===')
display(battery_df.drop_duplicates('model')[['model', 'auc']].set_index('model').round(4))

## 7. Per-Year Stability per Model (Premium Tier)

In [ ]:
test_df_copy = test_df.copy()
test_df_copy['year'] = test_df_copy.index.year

stability_rows = []
for model_name in ['LightGBM', 'XGBoost', 'CatBoost', 'Voting']:
    vp = val_probas[model_name]
    tp = test_probas[model_name]
    vs = np.sort(vp)[::-1]
    cutoff_prem = float(vs[max(1, int(len(vs) * 0.01) - 1)])
    test_df_copy['proba'] = tp
    yearly = {}
    for year, sub in test_df_copy.groupby('year'):
        mask = sub['proba'] >= cutoff_prem
        sub_l = sub.loc[mask, 'label']
        if len(sub_l) < 30:
            continue
        m = tier_metrics(sub_l)
        yearly[int(year)] = m
    pf_values = [m['pf'] for m in yearly.values() if not np.isinf(m['pf']) and m['pf'] > 0]
    if len(pf_values) >= 2:
        cv = float(np.std(pf_values) / np.mean(pf_values))
    else:
        cv = float('nan')
    for year, m in yearly.items():
        stability_rows.append({'model': model_name, 'year': year, 'pf': m['pf'], 'wr': m['wr'], 'n': m['n']})
    print(f'{model_name}: years tested = {sorted(yearly.keys())}, stability CV = {cv:.3f}')
    print(f'  yearly PFs: {[(y, round(m["pf"], 3)) for y, m in yearly.items()]}')

stab_df = pd.DataFrame(stability_rows)
if not stab_df.empty:
    print('\nPremium PF per year per model:')
    display(stab_df.pivot_table(index='year', columns='model', values='pf').round(4))

## 7.5 Trades pro Tag pro Modell × Tier

User-facing-relevant: wie viele Signale liefert jedes Modell pro Tag?


In [ ]:
test_days = (test_df.index.max() - test_df.index.min()).days
n_test_symbols = test_df['symbol'].nunique() if 'symbol' in test_df.columns else 1
print(f'TEST period: {test_days} days across {n_test_symbols} symbol(s)')

trades_per_day_rows = []
for r in rows:
    per_day_total = r['n'] / max(test_days, 1)
    per_day_per_symbol = per_day_total / max(n_test_symbols, 1)
    trades_per_day_rows.append({
        'model': r['model'],
        'tier': r['tier'],
        'total_trades': r['n'],
        'trades_per_day_all_symbols': per_day_total,
        'trades_per_day_per_symbol': per_day_per_symbol,
    })
tpd_df = pd.DataFrame(trades_per_day_rows)
print('\n=== Trades per Day per Tier per Model (TEST set, all symbols combined) ===')
display(tpd_df.pivot_table(index='model', columns='tier', values='trades_per_day_all_symbols')
        .round(2)[['Standard', 'High', 'Premium']])

print('\n=== Trades per Day PER SYMBOL ===')
display(tpd_df.pivot_table(index='model', columns='tier', values='trades_per_day_per_symbol')
        .round(2)[['Standard', 'High', 'Premium']])


## 7.7 GBPUSD Hold-Out Test pro Modell

**Wichtigster Test:** GBPUSD wurde im Training nie gesehen. Wenn die Edge generalisierungsfähig ist, sollte sie auch hier funktionieren.


In [ ]:
# Load GBPUSD extended features (held-out symbol)
ho_frames = []
for tf in PRIMARY_TIMEFRAMES:
    d = load_ext('GBPUSD', tf)
    if d is not None and not d.empty:
        ho_frames.append(d)
ho_df = pd.concat(ho_frames, axis=0).sort_index() if ho_frames else pd.DataFrame()

# Filter to OOS window only (>= VAL_END)
val_end_ts = pd.Timestamp(VAL_END)
if val_end_ts.tz is None:
    val_end_ts = val_end_ts.tz_localize('UTC')
else:
    val_end_ts = val_end_ts.tz_convert('UTC')

ho_df = ho_df[ho_df.index >= val_end_ts]
ho_df = ho_df.dropna(subset=available + ['label'])
print(f'GBPUSD hold-out OOS: {len(ho_df):,} bars')

ho_rows = []
ho_test_probas = {}
for model_name in ['LightGBM', 'XGBoost', 'CatBoost', 'Voting']:
    if model_name == 'LightGBM':
        ho_p = models['LightGBM'].predict(ho_df[available].values)
    elif model_name == 'Voting':
        ho_p = (models['LightGBM'].predict(ho_df[available].values) +
                 models['XGBoost'].predict_proba(ho_df[available].values)[:, 1] +
                 models['CatBoost'].predict_proba(ho_df[available].values)[:, 1]) / 3
    else:
        ho_p = models[model_name].predict_proba(ho_df[available].values)[:, 1]
    ho_test_probas[model_name] = ho_p

    # Use same VAL-derived cutoffs as the in-sample test (transferred)
    vp = val_probas[model_name]
    vs = np.sort(vp)[::-1]
    cutoff_std = float(vs[max(1, int(len(vs) * 0.10) - 1)])
    cutoff_high = float(vs[max(1, int(len(vs) * 0.03) - 1)])
    cutoff_prem = float(vs[max(1, int(len(vs) * 0.01) - 1)])

    for tier_name, cutoff in [('Standard', cutoff_std), ('High', cutoff_high), ('Premium', cutoff_prem)]:
        mask = ho_p >= cutoff
        sub_labels = ho_df['label'].iloc[mask.nonzero()[0]]
        m = tier_metrics(sub_labels)
        m['model'] = model_name
        m['tier'] = tier_name
        ho_rows.append(m)

ho_battery = pd.DataFrame(ho_rows)

print('\n=== GBPUSD Hold-Out Profit Factor ===')
display(ho_battery.pivot_table(index='model', columns='tier', values='pf')
        .round(4)[['Standard', 'High', 'Premium']])

print('\n=== GBPUSD Hold-Out Win Rate ===')
display(ho_battery.pivot_table(index='model', columns='tier', values='wr')
        .round(4)[['Standard', 'High', 'Premium']])

print('\n=== GBPUSD Hold-Out Trade Count ===')
display(ho_battery.pivot_table(index='model', columns='tier', values='n')
        .round(0)[['Standard', 'High', 'Premium']])


## 7.9 Consensus / Meta-Labeling Filter

**Idee:** Statt Voting (Mittelwert) als 4. Modell, teste Consensus: Premium-Signal **nur** wenn alle 3 Modelle (LGBM + XGB + CatBoost) zustimmen. Das ist Meta-Labeling auf Modell-Ebene: ein Modell sagt "Trade", die anderen filtern.

Wenn der Consensus-PF deutlich besser als das beste Einzelmodell ist, könnten wir es als Filter-Layer einbauen (auch wenn nur das beste Einzelmodell in Pine läuft).


In [ ]:
def per_model_cutoff(vp, pct):
    vs = np.sort(vp)[::-1]
    return float(vs[max(1, int(len(vs) * pct / 100) - 1)])

cutoffs_per_model = {}
for m in ['LightGBM', 'XGBoost', 'CatBoost']:
    cutoffs_per_model[m] = {
        'Standard': per_model_cutoff(val_probas[m], 10),
        'High':     per_model_cutoff(val_probas[m], 3),
        'Premium':  per_model_cutoff(val_probas[m], 1),
    }

# Consensus on in-sample TEST
consensus_rows = []
for tier_name in ['Standard', 'High', 'Premium']:
    mask = (
        (test_probas['LightGBM'] >= cutoffs_per_model['LightGBM'][tier_name]) &
        (test_probas['XGBoost']  >= cutoffs_per_model['XGBoost'][tier_name]) &
        (test_probas['CatBoost'] >= cutoffs_per_model['CatBoost'][tier_name])
    )
    sub_labels = test_df['label'].iloc[mask.nonzero()[0]]
    m = tier_metrics(sub_labels)
    m['filter'] = f'Consensus ({tier_name})'
    consensus_rows.append(m)

# Compare to LightGBM alone at same tiers
for tier_name in ['Standard', 'High', 'Premium']:
    mask = test_probas['LightGBM'] >= cutoffs_per_model['LightGBM'][tier_name]
    sub_labels = test_df['label'].iloc[mask.nonzero()[0]]
    m = tier_metrics(sub_labels)
    m['filter'] = f'LightGBM-only ({tier_name})'
    consensus_rows.append(m)

consensus_df = pd.DataFrame(consensus_rows)
print('Consensus vs LightGBM-only on TEST (in-sample symbols):')
display(consensus_df[['filter', 'n', 'wr', 'pf', 'expR']].round(4))

# Same on GBPUSD hold-out
ho_consensus_rows = []
for tier_name in ['Standard', 'High', 'Premium']:
    mask = (
        (ho_test_probas['LightGBM'] >= cutoffs_per_model['LightGBM'][tier_name]) &
        (ho_test_probas['XGBoost']  >= cutoffs_per_model['XGBoost'][tier_name]) &
        (ho_test_probas['CatBoost'] >= cutoffs_per_model['CatBoost'][tier_name])
    )
    sub_labels = ho_df['label'].iloc[mask.nonzero()[0]]
    m = tier_metrics(sub_labels)
    m['filter'] = f'Consensus ({tier_name})'
    ho_consensus_rows.append(m)
for tier_name in ['Standard', 'High', 'Premium']:
    mask = ho_test_probas['LightGBM'] >= cutoffs_per_model['LightGBM'][tier_name]
    sub_labels = ho_df['label'].iloc[mask.nonzero()[0]]
    m = tier_metrics(sub_labels)
    m['filter'] = f'LightGBM-only ({tier_name})'
    ho_consensus_rows.append(m)
ho_cons_df = pd.DataFrame(ho_consensus_rows)
print('\nConsensus vs LightGBM-only on GBPUSD hold-out:')
display(ho_cons_df[['filter', 'n', 'wr', 'pf', 'expR']].round(4))


## 8. Pine-Export-Eignung

In [ ]:
exportability = {
    'LightGBM': {'pine_ready': True,  'notes': 'Native tree-cascade, well-understood, our baseline'},
    'XGBoost':  {'pine_ready': True,  'notes': 'Similar tree structure, exportable with similar effort'},
    'CatBoost': {'pine_ready': False, 'notes': 'Oblivious trees + categorical embeddings — complex/impossible Pine export'},
    'Voting':   {'pine_ready': True,  'notes': 'Need to embed 3 models in Pine — ~3x line count, possible but bulky'},
}
exp_df = pd.DataFrame(exportability).T
print('Pine export feasibility per model:')
display(exp_df)

## 9. Decision Logic

In [ ]:
print('=' * 70)
print('MODEL BATTERY VERDICT')
print('=' * 70)

premium_pfs = battery_df[battery_df['tier'] == 'Premium'].set_index('model')['pf'].to_dict()
lgbm_pf = premium_pfs['LightGBM']
print(f'\nLightGBM baseline Premium PF: {lgbm_pf:.4f}')
print()
for name in ['XGBoost', 'CatBoost', 'Voting']:
    pf = premium_pfs[name]
    lift = pf - lgbm_pf
    pine_ready = exportability[name]['pine_ready']
    ready_str = 'Pine-ready' if pine_ready else 'Backend-only'
    print(f'  {name:<10s} PF {pf:.4f}  lift {lift:+.4f}  [{ready_str}]')

best_name = max(premium_pfs, key=premium_pfs.get)
best_pf = premium_pfs[best_name]
best_lift = best_pf - lgbm_pf
best_pine = exportability[best_name]['pine_ready']

print('\n' + '-' * 70)
print(f'Best model: {best_name} (Premium PF {best_pf:.4f}, lift {best_lift:+.4f})')
if best_name == 'LightGBM' or best_lift < 0.05:
    print('-> LightGBM remains the choice (lift too small to justify complexity)')
elif best_pine:
    print(f'-> {best_name} wins AND is Pine-exportable')
else:
    print(f'-> {best_name} wins but is NOT Pine-exportable')
    print(f'   Decision tree:')
    print(f'   - If V1 must be Pine-only: use 2nd-best Pine-ready model')
    print(f'   - If accept Backend-V1: use {best_name} via webhook → TradingView')

print('=' * 70)

## 10. Export Results to `/results/`

Schreibt alle Modell-Battery-Ergebnisse strukturiert in das Repo, damit sie versionierbar sind und in `/research/model_battery_results.md` ohne Re-Run referenziert werden können.

Was geschrieben wird:
- `json_exports/nb12_model_battery_{date}.json` — vollständiger Snapshot (Hyperparams, Metriken, Verdict)
- `benchmark_tables/nb12_pf_wr_expR_{date}.csv` — Modell × Tier PF/WR/ExpR (in-sample TEST)
- `per_symbol_metrics/nb12_gbpusd_holdout_{date}.csv` — GBPUSD Hold-Out pro Modell × Tier
- `yearly_stability_tables/nb12_premium_pf_by_year_{date}.csv` — PF pro Jahr × Modell (Premium-Tier)
- `benchmark_tables/nb12_consensus_filter_{date}.csv` — Consensus vs LightGBM-only Vergleich


In [ ]:
# ===== EXPORT — write structured results to /results/ for repo tracking =====

def _safe(x):
    """Make NaN/Inf JSON-serializable."""
    if isinstance(x, float):
        if np.isnan(x): return None
        if np.isinf(x): return 'inf' if x > 0 else '-inf'
    return x

def _df_to_records(df):
    """DataFrame to list of dicts, NaN/Inf-safe."""
    return [{k: _safe(v) for k, v in row.items()} for row in df.to_dict(orient='records')]

# --- 10.1 Benchmark table: in-sample TEST PF/WR/ExpR per model × tier ---
bench_path = RESULTS_DIR / 'benchmark_tables' / f'nb12_pf_wr_expR_{RUN_DATE}.csv'
battery_df[['model', 'tier', 'n', 'wins', 'losses', 'wr', 'pf', 'expR', 'cutoff', 'auc']].to_csv(bench_path, index=False)
print(f'[1/5] Wrote benchmark table -> {bench_path.relative_to(PROJECT_ROOT)}')

# --- 10.2 GBPUSD hold-out per model × tier ---
ho_path = RESULTS_DIR / 'per_symbol_metrics' / f'nb12_gbpusd_holdout_{RUN_DATE}.csv'
ho_battery[['model', 'tier', 'n', 'wins', 'losses', 'wr', 'pf', 'expR']].to_csv(ho_path, index=False)
print(f'[2/5] Wrote GBPUSD hold-out -> {ho_path.relative_to(PROJECT_ROOT)}')

# --- 10.3 Yearly stability (Premium PF per year per model) ---
stab_path = RESULTS_DIR / 'yearly_stability_tables' / f'nb12_premium_pf_by_year_{RUN_DATE}.csv'
if not stab_df.empty:
    stab_df.to_csv(stab_path, index=False)
    print(f'[3/5] Wrote yearly stability -> {stab_path.relative_to(PROJECT_ROOT)}')
else:
    print(f'[3/5] Skipped yearly stability (empty df)')

# --- 10.4 Consensus filter comparison ---
cons_path = RESULTS_DIR / 'benchmark_tables' / f'nb12_consensus_filter_{RUN_DATE}.csv'
combined_cons = pd.concat([
    consensus_df.assign(dataset='in_sample_TEST'),
    ho_cons_df.assign(dataset='gbpusd_holdout'),
], ignore_index=True)
combined_cons.to_csv(cons_path, index=False)
print(f'[4/5] Wrote consensus comparison -> {cons_path.relative_to(PROJECT_ROOT)}')

# --- 10.5 Full JSON snapshot (canonical reference for /research/model_battery_results.md) ---
premium_pfs = battery_df[battery_df['tier'] == 'Premium'].set_index('model')['pf'].to_dict()
lgbm_pf_premium = premium_pfs.get('LightGBM')
best_name = max(premium_pfs, key=lambda k: premium_pfs[k] if not np.isinf(premium_pfs[k]) else -1)
best_pf = premium_pfs[best_name]
best_lift = best_pf - lgbm_pf_premium if lgbm_pf_premium is not None else None

# Per-model stability CV (recomputed from stab_df)
stability_cv = {}
if not stab_df.empty:
    for m in stab_df['model'].unique():
        pfs = stab_df[stab_df['model'] == m]['pf']
        finite = pfs[~pfs.isin([np.inf, -np.inf])].dropna()
        if len(finite) >= 2:
            stability_cv[m] = float(finite.std() / finite.mean())
        else:
            stability_cv[m] = None

snapshot = {
    'notebook': 'NB12_model_battery',
    'run_date_utc': RUN_DATE,
    'random_seed': RANDOM_SEED,
    'lib_versions': {
        'lightgbm': lgb.__version__,
        'xgboost':  xgb.__version__,
        'catboost': catboost.__version__,
    },
    'config': {
        'asset_scope':       best_config['asset_scope'],
        'n_features':        best_config['n_features'],
        'feature_cols':      best_config['feature_cols'],
        'train_end':         str(TRAIN_END),
        'val_end':           str(VAL_END),
        'r_value':           R_VALUE,
        'win_r':             WIN_R,
        'loss_r':            LOSS_R,
        'nb11_baseline_pf':  best_config['premium_pf_oos'],
    },
    'dataset_shape': {
        'train_rows': int(len(train_df)),
        'val_rows':   int(len(val_df)),
        'test_rows':  int(len(test_df)),
        'gbpusd_holdout_rows': int(len(ho_df)),
        'class_balance_train': float(y_tr.mean()),
        'class_balance_val':   float(y_vl.mean()),
        'class_balance_test':  float(y_ts.mean()),
    },
    'hyperparams': {
        'LightGBM': {k: _safe(v) for k, v in lgb_params.items()},
        'XGBoost': {
            'n_estimators': 30, 'max_depth': 3, 'learning_rate': 0.05,
            'reg_lambda': 0.5, 'subsample': 0.85, 'colsample_bytree': 0.85,
            'scale_pos_weight': float(pos_weight), 'random_state': RANDOM_SEED,
        },
        'CatBoost': {
            'iterations': 30, 'depth': 3, 'learning_rate': 0.05,
            'l2_leaf_reg': 3.0, 'rsm': 0.85,
            'auto_class_weights': 'Balanced', 'random_seed': RANDOM_SEED,
        },
    },
    'in_sample_test_metrics': _df_to_records(battery_df),
    'gbpusd_holdout_metrics': _df_to_records(ho_battery),
    'yearly_stability_premium': _df_to_records(stab_df) if not stab_df.empty else [],
    'stability_cv_per_model': stability_cv,
    'consensus_vs_lgbm_only': {
        'in_sample_test': _df_to_records(consensus_df),
        'gbpusd_holdout': _df_to_records(ho_cons_df),
    },
    'pine_exportability': exportability,
    'verdict': {
        'lgbm_premium_pf':     _safe(lgbm_pf_premium),
        'best_model':          best_name,
        'best_premium_pf':     _safe(best_pf),
        'lift_over_lgbm':      _safe(best_lift),
        'lift_threshold':      0.05,
        'wins_by_threshold':   (best_lift is not None and best_lift >= 0.05 and best_name != 'LightGBM'),
        'best_is_pine_ready':  exportability[best_name]['pine_ready'],
    },
}

json_path = RESULTS_DIR / 'json_exports' / f'nb12_model_battery_{RUN_DATE}.json'
with open(json_path, 'w') as f:
    json.dump(snapshot, f, indent=2, default=str)
print(f'[5/5] Wrote full snapshot -> {json_path.relative_to(PROJECT_ROOT)}')

# --- Drive backup for results (Colab only) ---
if IS_COLAB:
    print('\nResults are already on Drive (RESULTS_DIR points to Drive). Pull from local clone to commit:')
    print('  git -C ~/Downloads/pace-algo pull origin main')
    print(f'  cp -r /content/drive/MyDrive/pace-algo/results/* ~/Downloads/pace-algo/results/')

print('\n' + '=' * 70)
print('EXPORT DONE. Files in /results/:')
print('=' * 70)
for sub in sorted((RESULTS_DIR).rglob(f'*{RUN_DATE}*')):
    print(f'  {sub.relative_to(RESULTS_DIR)}')
print(f'\nNext step: commit /results/ to GitHub so Claude can analyze without re-run.')


## 11. Auto-Push Results to GitHub (Optional)

Pusht die in Section 10 generierten `/results/`-Files direkt von Colab nach `github.com/ecoNC/pace-algo`. Damit muss Nico nichts manuell aus Drive runterladen — der Sibling-Claude auf dem anderen PC kann die Ergebnisse sofort lesen.

### One-Time Setup (nur beim ersten Run nötig)

**1. GitHub Personal Access Token erzeugen:**
- Gehe zu https://github.com/settings/personal-access-tokens
- "Generate new token" → **Fine-grained** token
- Repository access: **Only select repositories** → `ecoNC/pace-algo`
- Permissions → Repository permissions → **Contents: Read and write**
- Expiration: 90 Tage (oder länger)
- Generate → Token kopieren (wird nur einmal gezeigt)

**2. Token in Colab Secrets speichern:**
- Im Colab-Notebook: links das **🔑 Schlüssel-Icon** anklicken
- "Add new secret"
- Name: `GITHUB_TOKEN`
- Value: `<dein-Token>`
- Toggle "Notebook access" für dieses Notebook AN

**3. Diese Cell laufen lassen.** Beim nächsten Run automatisch.

### Was passiert wenn ich diese Cell laufen lasse?

- Clone des Repos in `/tmp/pace-algo-push/` (frisch, ohne Drive-State)
- Pull origin/main (rebase) — verhindert Konflikt mit Sibling-Claude
- Copy nur Files matching `*{RUN_DATE}*` aus Drive-`/results/` ins Repo
- Commit als **ecoNC** (Privacy-Lock per HANDOFF 12.4.19)
- Push origin/main
- Print Commit-SHA + GitHub-URL

### Was passiert NICHT?

- Keine Notebook-Outputs werden gepusht (Privacy + Diff-Größe)
- Keine `artifacts/models/*.pkl` werden gepusht (zu groß, lokal-only)
- Token wird NIE ins Notebook-File geschrieben (nur im RAM während dieser Cell)


In [ ]:
# ===== AUTO-PUSH results to GitHub (Colab only) =====
# Requires: GITHUB_TOKEN secret in Colab (see markdown above)
# Behavior on missing token: clear error, no silent fail

import shutil, subprocess
from pathlib import Path as _P

if not IS_COLAB:
    print('Local run — skip auto-push (files already in repo).')
else:
    try:
        from google.colab import userdata
        GH_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception as e:
        GH_TOKEN = None
        print(f'ERROR: cannot read GITHUB_TOKEN from Colab secrets: {e}')
        print('Setup: 🔑 icon (left sidebar) → Add secret → name=GITHUB_TOKEN, value=<your PAT>')
        print('Then enable "Notebook access" toggle for THIS notebook.')

    if GH_TOKEN:
        REPO_URL_HTTPS = 'github.com/ecoNC/pace-algo.git'
        AUTHOR_NAME    = 'ecoNC'
        AUTHOR_EMAIL   = 'ecoNC@users.noreply.github.com'
        TMP_REPO       = _P('/tmp/pace-algo-push')

        # 1. Fresh clone (avoid stale Drive state). Token in URL for this clone only.
        if TMP_REPO.exists():
            shutil.rmtree(TMP_REPO)
        clone_url = f'https://{GH_TOKEN}@{REPO_URL_HTTPS}'
        r = subprocess.run(['git', 'clone', '--quiet', clone_url, str(TMP_REPO)],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print(f'CLONE FAILED:\n  stderr: {r.stderr}')
            raise SystemExit('Stopped — fix token/repo access first.')
        print(f'[1/5] Cloned ecoNC/pace-algo into {TMP_REPO}')

        # 2. Configure committer identity (privacy: ecoNC only, never real name)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.name', AUTHOR_NAME], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.email', AUTHOR_EMAIL], check=True)

        # 3. Copy only files matching THIS run (RUN_DATE) from Drive results/
        target_results = TMP_REPO / 'results'
        copied = []
        for f in sorted(RESULTS_DIR.rglob(f'*{RUN_DATE}*')):
            if not f.is_file():
                continue
            rel = f.relative_to(RESULTS_DIR)
            dest = target_results / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied.append(rel)
        if not copied:
            print(f'[2/5] No files matching *{RUN_DATE}* found in {RESULTS_DIR} — nothing to push.')
            raise SystemExit('Nothing to commit.')
        print(f'[2/5] Copied {len(copied)} result files into clone:')
        for c in copied:
            print(f'        results/{c}')

        # 4. Pull --rebase first (sibling Claude may have pushed since clone — clone is fresh, but safe pattern)
        r = subprocess.run(['git', '-C', str(TMP_REPO), 'pull', '--rebase', '--quiet', 'origin', 'main'],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print(f'PULL --rebase FAILED:\n  stderr: {r.stderr}')
            print('Manual intervention needed — sibling Claude likely committed conflicting work.')
            raise SystemExit('Stopped.')
        print('[3/5] Pulled latest from origin/main (rebase, no conflicts)')

        # 5. Stage + commit + push
        subprocess.run(['git', '-C', str(TMP_REPO), 'add', 'results/'], check=True)
        r_status = subprocess.run(['git', '-C', str(TMP_REPO), 'status', '--porcelain'],
                                   capture_output=True, text=True)
        if not r_status.stdout.strip():
            print('[4/5] Nothing to commit (results already on origin).')
        else:
            commit_msg = f'NB12 results: model battery run {RUN_DATE}\n\nAuto-pushed from Colab. Random seed {RANDOM_SEED}. {len(copied)} files.'
            r_commit = subprocess.run(['git', '-C', str(TMP_REPO), 'commit', '-m', commit_msg],
                                       capture_output=True, text=True)
            if r_commit.returncode != 0:
                print(f'COMMIT FAILED:\n  stderr: {r_commit.stderr}')
                raise SystemExit('Stopped.')
            # Get SHA
            r_sha = subprocess.run(['git', '-C', str(TMP_REPO), 'rev-parse', '--short', 'HEAD'],
                                    capture_output=True, text=True)
            sha = r_sha.stdout.strip()
            print(f'[4/5] Committed as ecoNC: {sha}')

            r_push = subprocess.run(['git', '-C', str(TMP_REPO), 'push', 'origin', 'main'],
                                     capture_output=True, text=True)
            if r_push.returncode != 0:
                print(f'PUSH FAILED:\n  stderr: {r_push.stderr}')
                raise SystemExit('Stopped — push rejected.')
            print(f'[5/5] Pushed to origin/main')
            print(f'\n✓ DONE. View commit: https://github.com/ecoNC/pace-algo/commit/{sha}')
            print(f'✓ Files now visible at: https://github.com/ecoNC/pace-algo/tree/main/results')

        # Cleanup — never leave the token-bearing clone behind
        shutil.rmtree(TMP_REPO)


---

Schick mir nach dem Run:
- Section 6 (PF/WR/Expectancy pro Modell × Tier + AUC)
- Section 7 (Per-Year Stability Tabelle)
- Section 8 (Pine-Export-Eignung)
- Section 9 (Verdict)

Damit treffen wir die finale Architekturentscheidung vor V1.